# Mekanisme Data Curriculum Learning per Stage 80/20 + Lampiran Split

Notebook ini membuat mekanisme data dari awal untuk eksperimen **CNN klasik vs Hybrid QCQ-CNN**.

Tujuan utama:
1. Membaca `dataset_final_manifest.csv` dari notebook EDA/deduplikasi.
2. Membagi data menjadi `development` dan `holdout_test` dengan stratified split 90/10.
3. Membuat subset data per stage curriculum learning.
4. Membagi subset stage menjadi train/validation dengan prinsip **80/20**.
   - Stage holdout memakai `val_size=0.20`.
   - Stage K-Fold memakai 5-Fold, sehingga 4 fold train dan 1 fold validation = 80/20.
5. Menyimpan subset **training natural** dan **validation natural** per stage/fold untuk lampiran dan pertanggungjawaban.
6. Menerapkan balancing dan augmentasi hanya pada training subset.
7. Menjaga validation dan holdout test tetap natural untuk mencegah data leakage.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_project_root(marker="dataset_final_manifest.csv"):
    """Cari root repo dari current working directory notebook."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Root proyek dengan {marker!r} tidak ditemukan dari {start} atau parent-nya.")

PROJECT_ROOT = find_project_root()
UTILS_PATH = PROJECT_ROOT / "02_curriculum_pipeline"
if str(UTILS_PATH.resolve()) not in sys.path:
    sys.path.append(str(UTILS_PATH.resolve()))

from curriculum_stage_utils import (
    DEFAULT_STAGE_CONFIGS,
    StageConfig,
    load_manifest,
    save_manifest,
    class_distribution,
    stratified_global_split,
    build_all_stage_assets,
    summarize_stage_outputs,
    summarize_train_validation_subset_indices,
    make_stage_table,
    get_stage_train_plan,
    get_stage_validation_manifest,
    get_stage_natural_train_manifest,
    get_stage_natural_validation_manifest,
    materialize_augmentation_plan,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


## 1. Konfigurasi utama

`dataset_final_manifest.csv` wajib dihasilkan oleh notebook pertama setelah:
- audit dataset awal,
- deduplikasi tahap pertama,
- preprocessing,
- deduplikasi tahap kedua,
- audit final.

Folder output curriculum akan menyimpan semua CSV split dan augmentation plan.


In [ ]:
# =========================
# CONFIG
# =========================

MANIFEST_PATH = PROJECT_ROOT / "dataset_final_manifest.csv"
OUTPUT_DIR = PROJECT_ROOT / "curriculum_outputs"

GLOBAL_DEV_SIZE = 0.90
RANDOM_STATE = 42
MAX_UNIQUE_PER_CLASS = 210
FINAL_TARGET_PER_CLASS = 1000  # 10 kelas × 1.000 = 10.000 citra training augmented

# Mode balancing:
# - equal_unique_cap: semua kelas training diambil sama dengan min(210, jumlah kelas terkecil).
# - cap_only: kelas besar dibatasi 210, kelas kecil dipakai seluruhnya.
BALANCE_MODE = "equal_unique_cap"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Stage configuration

Progresi yang digunakan:

| Stage | % development | Rasio natural train/validation | Target augmentasi | Total 10 kelas |
|---|---:|---:|---:|---:|
| Stage 1 | 5% | 80/20 | 5% dari 10.000 | 500 |
| Stage 2 | 25% | 80/20 | 25% dari 10.000 | 2.500 |
| Stage 3 | 50% | 80/20 via 5-Fold | 50% dari 10.000 | 5.000 |
| Stage 4 | 100% | 80/20 via Repeated 5-Fold | 100% dari 10.000 | 10.000 |
| Stage 5 | 100% | 80/20 | 100% dari 10.000 | 10.000 |

Jadi, **upscale ke 50% dari 10.000** sejalan untuk Stage 3, karena Stage 3 berfungsi sebagai tuning konvergensi dengan beban data sedang. Stage 4 dan Stage 5 memakai target penuh 10.000 sebagai augmented training set.

Catatan penting: rasio 80/20 dihitung pada **data natural sebelum augmentasi**. Setelah augmentasi, jumlah training akan jauh lebih besar daripada validation, dan itu memang sengaja karena validation tidak boleh diaugmentasi.


In [ ]:
# Jika ingin mengubah progresi, edit daftar ini.
stage_configs = [
    StageConfig(
        stage_id=1,
        stage_name="Sanity Test",
        dev_fraction=0.05,
        epoch=2,
        cv_mode="holdout",
        val_size=0.20,
        final_augmented_per_class=FINAL_TARGET_PER_CLASS,
        augmentation_fraction_of_final=0.05,
        max_unique_per_class=MAX_UNIQUE_PER_CLASS,
        random_state=RANDOM_STATE,
    ),
    StageConfig(
        stage_id=2,
        stage_name="Warm Start",
        dev_fraction=0.25,
        epoch=10,
        cv_mode="holdout",
        val_size=0.20,
        final_augmented_per_class=FINAL_TARGET_PER_CLASS,
        augmentation_fraction_of_final=0.25,
        max_unique_per_class=MAX_UNIQUE_PER_CLASS,
        random_state=RANDOM_STATE,
    ),
    StageConfig(
        stage_id=3,
        stage_name="Tuning for Convergence",
        dev_fraction=0.50,
        epoch=25,
        cv_mode="kfold",
        n_splits=5,
        n_repeats=1,
        final_augmented_per_class=FINAL_TARGET_PER_CLASS,
        augmentation_fraction_of_final=0.50,
        max_unique_per_class=MAX_UNIQUE_PER_CLASS,
        random_state=RANDOM_STATE,
    ),
    StageConfig(
        stage_id=4,
        stage_name="Tuning for Max Accuracy",
        dev_fraction=1.00,
        epoch=50,
        cv_mode="kfold",
        n_splits=5,
        n_repeats=1,
        final_augmented_per_class=FINAL_TARGET_PER_CLASS,
        augmentation_fraction_of_final=1.00,
        max_unique_per_class=MAX_UNIQUE_PER_CLASS,
        random_state=RANDOM_STATE,
    ),
    StageConfig(
        stage_id=5,
        stage_name="Final Repeated K-Fold Evaluation",
        dev_fraction=1.00,
        epoch=100,
        cv_mode="repeated_kfold",
        n_splits=5,
        n_repeats=5,
        final_augmented_per_class=FINAL_TARGET_PER_CLASS,
        augmentation_fraction_of_final=1.00,
        max_unique_per_class=MAX_UNIQUE_PER_CLASS,
        random_state=RANDOM_STATE,
    ),
]

stage_table = make_stage_table(stage_configs)
stage_table.to_csv(OUTPUT_DIR / "stage_configuration.csv", index=False)
stage_table


## 3. Load manifest final dan cek distribusi kelas

Jika file `dataset_final_manifest.csv` belum tersedia, jalankan notebook dataset awal terlebih dahulu.


In [ ]:
df = load_manifest(MANIFEST_PATH)
print("Total dataset final:", len(df))
display(class_distribution(df))

plt.figure(figsize=(10, 4))
dist = class_distribution(df)
plt.bar(dist["class"], dist["count"])
plt.xticks(rotation=45, ha="right")
plt.title("Distribusi Kelas Dataset Final Setelah Deduplikasi")
plt.xlabel("Kelas")
plt.ylabel("Jumlah Citra")
plt.tight_layout()
plt.show()


## 4. Global stratified split 90/10

Output:
- `development_manifest.csv`
- `holdout_test_manifest.csv`

Holdout test tidak boleh ikut balancing, augmentasi, tuning, atau pemilihan model.


In [ ]:
development_df, holdout_test_df = stratified_global_split(
    df,
    dev_size=GLOBAL_DEV_SIZE,
    random_state=RANDOM_STATE,
)

save_manifest(development_df, OUTPUT_DIR / "development_manifest.csv")
save_manifest(holdout_test_df, OUTPUT_DIR / "holdout_test_manifest.csv")

print("Development:", len(development_df))
display(class_distribution(development_df))

print("Holdout test:", len(holdout_test_df))
display(class_distribution(holdout_test_df))


## 5. Build curriculum stage assets

Setiap stage akan menghasilkan folder:

```text
curriculum_outputs/stage_01/
curriculum_outputs/stage_02/
curriculum_outputs/stage_03/
curriculum_outputs/stage_04/
curriculum_outputs/stage_05/
```

Masing-masing folder berisi:
- `stage_subset_manifest.csv`
- `stage_split_or_fold_assignment.csv`
- `train_validation_subset_index.csv`
- `train_validation_distribution_summary.csv`
- `train_validation_subsets/train_natural_stage_XX_repeat_YY_fold_ZZ.csv`
- `train_validation_subsets/validation_natural_stage_XX_repeat_YY_fold_ZZ.csv`
- `balanced_train_original_manifest.csv`
- `augmentation_plan.csv`
- `combined_train_plan_original_plus_augmented.csv`
- `stage_metadata.csv`

File `train_validation_subsets/` dipakai untuk **lampiran dan pertanggungjawaban split data**. File ini berisi subset natural sebelum balancing dan augmentasi.


In [ ]:
saved_outputs = build_all_stage_assets(
    dev_df=development_df,
    out_dir=OUTPUT_DIR,
    stage_configs=stage_configs,
    balance_mode=BALANCE_MODE,
    random_state=RANDOM_STATE,
)

summary = summarize_stage_outputs(OUTPUT_DIR)
summary.to_csv(OUTPUT_DIR / "all_stage_metadata_summary.csv", index=False)

subset_index = summarize_train_validation_subset_indices(OUTPUT_DIR)
subset_index.to_csv(OUTPUT_DIR / "all_stage_train_validation_subset_index.csv", index=False)

display(summary)
display(subset_index.head())


## 6. Cek target upscale per stage

Interpretasi:
- Stage 1 target 50 citra/kelas = 500 total untuk 10 kelas.
- Stage 2 target 250 citra/kelas = 2.500 total.
- Stage 3 target 500 citra/kelas = 5.000 total, yaitu 50% dari 10.000.
- Stage 4 dan 5 target 1.000 citra/kelas = 10.000 total.

Jika data unik pada training subset tidak cukup untuk target 210 per kelas, mekanisme akan memakai target adaptif:
`min(210, jumlah kelas terkecil pada training subset)`.


In [ ]:
stage_summary = (
    summary.groupby(["stage_id", "stage_name", "cv_mode", "augmented_target_per_class"], as_index=False)
    .agg(
        n_units=("fold_id", "count"),
        mean_unique_after_balance=("n_train_unique_after_balance", "mean"),
        mean_augmented_rows=("n_augmented_rows", "mean"),
        mean_total_train_after_aug=("n_train_total_after_augmentation_plan", "mean"),
    )
)

stage_summary["target_total_10_classes"] = stage_summary["augmented_target_per_class"] * 10
stage_summary


## 7. Audit subset train/validation natural untuk lampiran

Bagian ini memeriksa bahwa setiap stage/fold sudah menyimpan subset training dan validation natural. File ini penting untuk lampiran karena menunjukkan data mana yang dipakai sebagai train dan validation sebelum balancing/augmentasi.


In [ ]:
subset_index = pd.read_csv(OUTPUT_DIR / "all_stage_train_validation_subset_index.csv")
display(subset_index)

# Ringkasan ukuran natural train/validation per stage.
subset_size_summary = (
    subset_index.groupby("stage_id", as_index=False)
    .agg(
        n_units=("fold_id", "count"),
        mean_train_natural=("n_train_natural", "mean"),
        mean_validation_natural=("n_validation_natural", "mean"),
        min_train_natural=("n_train_natural", "min"),
        min_validation_natural=("n_validation_natural", "min"),
        max_train_natural=("n_train_natural", "max"),
        max_validation_natural=("n_validation_natural", "max"),
    )
)
subset_size_summary.to_csv(OUTPUT_DIR / "train_validation_size_summary_for_appendix.csv", index=False)
subset_size_summary


## 8. Cara memanggil dari notebook training lain

Contoh untuk mengambil training plan dan validation manifest pada stage tertentu.


In [ ]:
# Contoh: ambil Stage 3, Fold 0
stage_id = 3
fold_id = 0
repeat_id = 0

# Natural split untuk lampiran/audit
train_natural = get_stage_natural_train_manifest(OUTPUT_DIR, stage_id=stage_id, fold_id=fold_id, repeat_id=repeat_id)
val_natural = get_stage_natural_validation_manifest(OUTPUT_DIR, stage_id=stage_id, fold_id=fold_id, repeat_id=repeat_id)

# Training plan setelah balancing + augmentation plan
train_plan = get_stage_train_plan(OUTPUT_DIR, stage_id=stage_id, fold_id=fold_id, repeat_id=repeat_id)
val_manifest = get_stage_validation_manifest(OUTPUT_DIR, stage_id=stage_id, fold_id=fold_id, repeat_id=repeat_id)

print("Train natural sebelum balancing/augmentasi:", train_natural.shape)
print("Validation natural:", val_natural.shape)
print("Train plan setelah balancing/upscale:", train_plan.shape)
print("Validation dari assignment:", val_manifest.shape)

display(train_natural.head())
display(val_natural.head())
display(train_plan.head())


### Contoh pseudocode di PyTorch Dataset

Kolom penting:
- `path`: path gambar original.
- `is_augmented`: apakah baris ini augmented sample.
- `augmentation_ops`: operasi augmentasi, contoh `rotate_small+brightness_small`.
- `planned_output_path`: path rencana jika augmentasi dimaterialisasi offline.

Untuk online augmentation, Dataset bisa membaca `augmentation_ops` dan menerapkannya saat `__getitem__`.


In [ ]:
# Contoh sederhana pemakaian pada training notebook:
print('''
from curriculum_stage_utils import (
    get_stage_train_plan,
    get_stage_validation_manifest,
    get_stage_natural_train_manifest,
    get_stage_natural_validation_manifest,
)

# Untuk lampiran/audit split natural
train_natural = get_stage_natural_train_manifest("curriculum_outputs", stage_id=3, fold_id=0, repeat_id=0)
val_natural = get_stage_natural_validation_manifest("curriculum_outputs", stage_id=3, fold_id=0, repeat_id=0)

# Untuk training aktual
train_df = get_stage_train_plan("curriculum_outputs", stage_id=3, fold_id=0, repeat_id=0)
val_df = get_stage_validation_manifest("curriculum_outputs", stage_id=3, fold_id=0, repeat_id=0)

# train_df dipakai untuk training dataset
# val_df dipakai untuk validation dataset
# hanya train_df yang punya augmented rows
''')


## 9. Opsional: materialisasi augmentasi offline

Bagian ini opsional. Jika training memakai online augmentation di PyTorch Dataset, tidak perlu menjalankan cell ini.

Jika ingin benar-benar membuat file gambar augmentasi, jalankan `materialize_augmentation_plan`.


In [ ]:
# CONTOH OPSIONAL - jangan dijalankan kalau belum siap membuat file gambar augmentasi.
# aug_plan_path = OUTPUT_DIR / "stage_03" / "augmentation_plan.csv"
# materialized = materialize_augmentation_plan(
#     aug_plan_path,
#     output_root=Path("."),
#     overwrite=False,
#     mode="L",  # grayscale
# )
# materialized.to_csv(OUTPUT_DIR / "stage_03" / "augmentation_materialized_manifest.csv", index=False)
# materialized.head()


## 10. Leakage guard

Cell ini memeriksa bahwa `source_image_id` pada holdout test tidak muncul di development set. Karena augmentasi hanya dibuat dari training subset, validation dan holdout test tetap natural.


In [ ]:
dev_sources = set(development_df["source_image_id"].astype(str))
test_sources = set(holdout_test_df["source_image_id"].astype(str))
overlap = dev_sources.intersection(test_sources)

print("Overlap source_image_id development vs holdout:", len(overlap))
assert len(overlap) == 0, "Leakage terdeteksi antara development dan holdout test!"


## 11. Ringkasan keputusan

Keputusan workflow:
1. Dataset final dibagi dulu menjadi development dan holdout test.
2. Curriculum learning hanya memakai development set.
3. Tiap stage mengambil subset development secara stratified.
4. Seluruh stage memakai prinsip train/validation natural **80/20**.
   - Holdout internal memakai `val_size=0.20`.
   - K-Fold memakai 5 fold, sehingga 4 fold train dan 1 fold validation.
5. Subset train dan validation natural disimpan per stage/fold untuk lampiran.
6. Balancing adaptif hanya diterapkan pada training subset.
7. Augmentasi/upscale hanya diterapkan pada training subset.
8. Validation dan holdout test tidak disentuh augmentasi.
9. Upscale 50% dari 10.000 cocok untuk Stage 3; target 10.000 penuh dipakai pada Stage 4 dan Stage 5.
